In [ ]:
import cv2
import numpy as np
import os
import csv
from collections import deque

# =======================
# ----- CONFIGURE -------
# =======================

video_path = r"XXX.mp4"

video_name = os.path.splitext(os.path.basename(video_path))[0]
video_dir  = os.path.dirname(video_path)

output_csv        = os.path.join(video_dir, f"{video_name}_list.csv")
roi_reference_path = os.path.join(video_dir, f"{video_name}_roi_reference.jpg")

# "full"   = process entire video
# "frames" = process only from START_FRAME to END_FRAME
PROCESS_MODE = "full"

START_FRAME = 7000
END_FRAME   = 9200

ROI_SELECTION_FRAME = 7000
ENABLE_MANUAL_ROI   = True

# =======================
# --- ROTATION PARAMS ---
# =======================
# Step 1 (always first): an interactive picker shows the RAW full frame.
# Drag the trackbar until the road/horizon is level, then press ENTER.
# The confirmed angle is applied to EVERY frame before anything else runs.
# Set ENABLE_ROTATION_PICKER = False and hardcode FRAME_ROTATION_ANGLE
# if you already know the value.
ENABLE_ROTATION_PICKER = True
FRAME_ROTATION_ANGLE   = 0.0   # degrees; positive = clockwise

# =======================
# --- PKW PARAMETERS ----
# =======================

MIN_CONTOUR_AREA   = 20000
DILATE_ITERS       = 2
VAR_THRESHOLD      = 35
MAX_ASSOCIATION_DIST = 180
IOU_MATCH_THRESHOLD  = 0.05
MAX_MISSES = 25
SMOOTHING  = 8

LINE_POSITION_RATIO = 0.50

ROI_X_MIN_RATIO = 0.05
ROI_X_MAX_RATIO = 0.98
ROI_Y_MIN_RATIO = 0.20
ROI_Y_MAX_RATIO = 0.60

MIN_BOX_WIDTH       = 150
MIN_BOX_HEIGHT      = 80
MIN_BOX_AREA        = 20000
MAX_BOX_WIDTH_RATIO  = 0.85
MAX_BOX_HEIGHT_RATIO = 0.65

MERGE_X_GAP = 180
MERGE_Y_GAP = 80

ENABLE_LOW_ZONE_FILTER = True
LOW_ZONE_Y_RATIO = 0.75

ENABLE_ASPECT_RATIO_FILTER = True
MIN_ASPECT_RATIO = 0.8
MAX_ASPECT_RATIO = 5.0

# =======================
# --- LKW PARAMETERS ----
# =======================

LKW_FILL_THRESHOLD      = 0.55
LKW_COOLDOWN_FRAMES     = 45

LKW_MIN_BOX_WIDTH_RATIO  = 0.35
LKW_MIN_BOX_HEIGHT_RATIO = 0.20
LKW_MAX_BOX_WIDTH_RATIO  = 0.99
LKW_MAX_BOX_HEIGHT_RATIO = 0.98
LKW_MIN_ASPECT_RATIO     = 1.2
LKW_MAX_ASPECT_RATIO     = 10.0
LKW_MIN_CONTOUR_AREA     = 40000
LKW_MERGE_X_GAP          = 200
LKW_MERGE_Y_GAP          = 120

# =======================
# --- CONTROL MODE ------
# =======================

PLAYBACK_DELAY_MS = 80

SAVE_EMPTY_CSV            = True
METERS_PER_PIXEL          = 0.010309
MAX_REALISTIC_SPEED_KMH   = 200
PROCESS_EVERY_N_FRAMES    = 1
FORCE_SAVE_EACH_LOG_TO_DISK = True

MAX_DISPLAY_WIDTH  = 1600
MAX_DISPLAY_HEIGHT = 900

RECENT_CROSSINGS_SHOWN = 6


# =======================
# ----- UTILITIES -------
# =======================

def iou(a, b):
    ax1, ay1, aw, ah = a;  ax2, ay2 = ax1 + aw, ay1 + ah
    bx1, by1, bw, bh = b;  bx2, by2 = bx1 + bw, by1 + bh
    iw = max(0, min(ax2, bx2) - max(ax1, bx1))
    ih = max(0, min(ay2, by2) - max(ay1, by1))
    inter = iw * ih
    union = aw * ah + bw * bh - inter + 1e-6
    return inter / union


def box_center(b):
    x, y, w, h = b
    return x + w / 2.0, y + h / 2.0


def format_hms(t):
    h = int(t // 3600);  m = int((t % 3600) // 60);  s = int(t % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def crossing_label_from_dx(dx_val):
    return "L2R" if dx_val > 0 else "R2L"


def get_display_scale(w, h, max_w=1600, max_h=900):
    return min(max_w / w, max_h / h, 1.0)


def resize_for_display(img, max_w=1600, max_h=900):
    h, w = img.shape[:2]
    s = get_display_scale(w, h, max_w, max_h)
    if s >= 1.0:
        return img, s
    return cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA), s


# =======================
# --- ROTATION HELPERS --
# =======================

def build_rotation_matrix(src_w, src_h, angle_deg):
    """Return (M_2x3, new_w, new_h) for rotating a src_w×src_h image by angle_deg."""
    cx, cy = src_w / 2.0, src_h / 2.0
    M = cv2.getRotationMatrix2D((cx, cy), -angle_deg, 1.0)
    cos_a, sin_a = abs(M[0, 0]), abs(M[0, 1])
    new_w = int(src_h * sin_a + src_w * cos_a)
    new_h = int(src_h * cos_a + src_w * sin_a)
    # shift so the full rotated image stays in frame
    M[0, 2] += new_w / 2.0 - cx
    M[1, 2] += new_h / 2.0 - cy
    return M, new_w, new_h


def apply_rotation(img, M, new_w, new_h):
    return cv2.warpAffine(img, M, (new_w, new_h),
                          flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_REPLICATE)


# =======================
# --- ROTATION PICKER ---
# =======================

def pick_rotation_angle(cap, frame_number, orig_w, orig_h):
    """
    Show the FULL raw frame in an interactive window.
    User drags a trackbar to level the horizon, then presses ENTER.
    Returns the confirmed angle in degrees.
    """
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    target = max(0, min(frame_number, total - 1))
    cap.set(cv2.CAP_PROP_POS_FRAMES, target)
    ok, raw_frame = cap.read()
    if not ok:
        raise RuntimeError(f"Could not read frame {target} for rotation picker.")

    win = "Step 1 — Rotation Picker: level the horizon, then press ENTER  (ESC = skip)"
    cv2.namedWindow(win, cv2.WINDOW_NORMAL)

    # Trackbar: 0–400  →  -20.0 … +20.0 deg  (centre 200 = 0 deg, step 0.1 deg)
    OFFSET, SCALE = 200, 10.0
    cv2.createTrackbar("Angle ×0.1 deg  (200 = 0°)", win, OFFSET, 400, lambda v: None)

    confirmed = 0.0

    while True:
        tb  = cv2.getTrackbarPos("Angle ×0.1 deg  (200 = 0°)", win)
        ang = (tb - OFFSET) / SCALE

        M, nw, nh = build_rotation_matrix(orig_w, orig_h, ang)
        rotated = apply_rotation(raw_frame, M, nw, nh)

        preview = rotated.copy()
        ph, pw = preview.shape[:2]

        # horizontal guide line at 1/3 and 2/3 height to judge level
        for frac in (1/3, 2/3):
            y_guide = int(ph * frac)
            cv2.line(preview, (0, y_guide), (pw, y_guide), (0, 255, 255), 1)

        cv2.putText(preview, f"Rotation: {ang:+.1f} deg", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 0, 0), 4)
        cv2.putText(preview, f"Rotation: {ang:+.1f} deg", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 255, 0), 2)
        cv2.putText(preview, "ENTER = confirm   ESC = skip (0.0 deg)", (20, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.75, (200, 200, 200), 2)
        cv2.putText(preview, "Cyan lines are reference guides — make road/horizon parallel",
                    (20, 115), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 1)

        disp, _ = resize_for_display(preview, MAX_DISPLAY_WIDTH, MAX_DISPLAY_HEIGHT)
        cv2.imshow(win, disp)

        key = cv2.waitKey(30) & 0xFF
        if key == 13:          # ENTER
            confirmed = ang
            print(f"[Rotation] Confirmed: {confirmed:+.1f} degrees")
            break
        elif key == 27:        # ESC
            confirmed = 0.0
            print("[Rotation] Skipped — using 0.0 degrees.")
            break

    cv2.destroyWindow(win)
    return confirmed


# =======================
# --- ROI SELECTION -----
# =======================

def select_roi_from_rotated_frame(cap, frame_number, M_rot, rot_w, rot_h):
    """Read a frame, rotate it with the confirmed transform, then let the user draw ROI."""
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    target = max(0, min(frame_number, total - 1))
    cap.set(cv2.CAP_PROP_POS_FRAMES, target)
    ok, raw = cap.read()
    if not ok:
        raise RuntimeError(f"Could not read frame {target} for ROI selection.")

    rotated = apply_rotation(raw, M_rot, rot_w, rot_h)

    preview = rotated.copy()
    cv2.putText(preview, f"Step 2 — Select ROI on ROTATED frame {target}, then press ENTER/SPACE",
                (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.85, (0, 255, 255), 2)
    cv2.putText(preview, "ESC to cancel", (20, 75),
                cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 255, 255), 2)

    disp, scale = resize_for_display(preview, MAX_DISPLAY_WIDTH, MAX_DISPLAY_HEIGHT)
    roi = cv2.selectROI("Manual ROI Selection (rotated frame)", disp,
                        showCrosshair=True, fromCenter=False)
    cv2.destroyWindow("Manual ROI Selection (rotated frame)")

    x, y, w, h = map(int, roi)
    if w <= 0 or h <= 0:
        raise ValueError("Invalid ROI or ROI selection cancelled.")

    # Scale back to full rotated-frame coords
    return (int(x / scale), int(y / scale),
            int(w / scale), int(h / scale),
            rotated, target)


# =======================
# ----- MORE UTILS ------
# =======================

def save_roi_reference_image(rotated_frame, roi_x_min, roi_y_min, roi_x_max, roi_y_max,
                              line_x, out_path, frame_number, angle):
    ref = rotated_frame.copy()
    cv2.rectangle(ref, (roi_x_min, roi_y_min), (roi_x_max, roi_y_max), (100, 255, 255), 3)
    cv2.line(ref, (line_x, 0), (line_x, ref.shape[0]), (255, 0, 0), 2)
    cv2.putText(ref, f"ROI reference — frame {frame_number}  rotation={angle:+.1f}°",
                (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
    cv2.putText(ref, os.path.basename(video_path),
                (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)
    cv2.imwrite(out_path, ref)


def write_log_row(csv_writer, csv_file, row):
    csv_writer.writerow(row)
    csv_file.flush()
    if FORCE_SAVE_EACH_LOG_TO_DISK:
        os.fsync(csv_file.fileno())


def draw_hud(vis, current_frame, end_frame, t, log_count, paused,
             playback_delay, mask_overlay_on, is_lkw_frame,
             mask_fill_ratio, lkw_cooldown, recent_crossings,
             rot_w, rotation_angle):

    status     = "PAUSED" if paused else f"{1000/max(playback_delay,1):.1f} fps ({playback_delay} ms)"
    lkw_tag    = "  *** LKW ***" if is_lkw_frame else ""
    cool_tag   = f"  cooldown={lkw_cooldown}" if lkw_cooldown > 0 else ""

    lines = [
        f"Frame: {current_frame}/{end_frame}   Time: {format_hms(t)}",
        f"Speed: {status}",
        f"Fill: {mask_fill_ratio:.2f}{lkw_tag}{cool_tag}",
        f"Crossings logged: {log_count}",
        f"Rotation: {rotation_angle:+.1f}°   Mask: {'ON' if mask_overlay_on else 'OFF'} (M)",
    ]
    for i, line in enumerate(lines):
        color = (0, 80, 255) if is_lkw_frame and i == 2 else (255, 255, 255)
        cv2.putText(vis, line, (20, 35 + i * 28), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0,0,0), 4)
        cv2.putText(vis, line, (20, 35 + i * 28), cv2.FONT_HERSHEY_SIMPLEX, 0.75, color, 2)

    x_off = max(20, rot_w - 460)
    cv2.putText(vis, "Recent crossings:", (x_off, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0,0,0), 4)
    cv2.putText(vis, "Recent crossings:", (x_off, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255,220,80), 2)
    for i, entry in enumerate(recent_crossings):
        cv2.putText(vis, entry, (x_off, 65 + i*26), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (0,0,0), 3)
        cv2.putText(vis, entry, (x_off, 65 + i*26), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (255,200,140), 2)

    help_text = "SPACE=pause/resume  N=step  +/-=speed  M=mask  S=snapshot  ESC=quit"
    cv2.putText(vis, help_text, (20, vis.shape[0]-12), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,0), 3)
    cv2.putText(vis, help_text, (20, vis.shape[0]-12), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 1)


# =======================
# ------ TRACK ----------
# =======================

class Track:
    def __init__(self, tid, box, vehicle_type="PKW"):
        self.id           = tid
        self.box_hist     = deque(maxlen=SMOOTHING)
        self.box_hist.append(box)
        self.misses       = 0
        self.crossed      = False
        self.vehicle_type = vehicle_type

    @property
    def box(self):    return self.box_hist[-1]
    @property
    def center(self): return box_center(self.box)

    def update(self, box):
        self.box_hist.append(box);  self.misses = 0

    def dx(self):
        if len(self.box_hist) < 2: return 0.0
        cx0, _ = box_center(self.box_hist[0])
        cx1, _ = box_center(self.box_hist[-1])
        return (cx1 - cx0) / max(1, len(self.box_hist) - 1)

    def leading_edge_x(self):
        x, y, w, h = self.box
        return x if self.dx() < 0 else x + w


# =======================
# --- BOX UTILITIES -----
# =======================

def should_merge(a, b, xg, yg):
    ax, ay, aw, ah = a;  bx, by, bw, bh = b
    return (ax < bx+bw+xg and ax+aw+xg > bx and
            ay < by+bh+yg and ay+ah+yg > by)


def merge_boxes(boxes, x_gap, y_gap):
    merged = []
    for det in boxes:
        placed = False
        for i, m in enumerate(merged):
            if should_merge(det, m, x_gap, y_gap):
                x1 = min(det[0], m[0]);  y1 = min(det[1], m[1])
                x2 = max(det[0]+det[2], m[0]+m[2])
                y2 = max(det[1]+det[3], m[1]+m[3])
                merged[i] = (x1, y1, x2-x1, y2-y1)
                placed = True;  break
        if not placed:
            merged.append(det)
    return merged


def choose_detection_for_track(track, detections, used):
    # try IoU first
    best_j, best_s = -1, -1.0
    for j, det in enumerate(detections):
        if j in used: continue
        s = iou(track.box, det)
        if s > best_s: best_s, best_j = s, j
    if best_j >= 0 and best_s >= IOU_MATCH_THRESHOLD:
        return best_j
    # fall back to centroid distance
    tcx, tcy = track.center
    best_j, best_d = -1, 1e9
    for j, det in enumerate(detections):
        if j in used: continue
        dcx, dcy = box_center(det)
        d = np.hypot(dcx-tcx, dcy-tcy)
        if d < best_d: best_d, best_j = d, j
    return best_j if best_j >= 0 and best_d <= MAX_ASSOCIATION_DIST else -1


# =======================
# ------- MAIN ----------
# =======================

# columns = ["TrackID","Frame","Timestamp_s","Timestamp_hms",
#            "Direction","VehicleType","center_x","y_center","Speed_kmh"]

columns = ["TrackID","Frame","Timestamp_s","Timestamp_hms","Timestamp_hms_us",
           "Direction","VehicleType","center_x","y_center","Speed_kmh"]

if not os.path.exists(video_path):
    raise FileNotFoundError(f"Video not found:\n{video_path}")

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise RuntimeError("Could not open video file.")

csv_file = None

try:
    fps         = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration    = frame_count / max(fps, 1e-6)
    orig_w      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_h      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # =========================================================
    # STEP 1 — ROTATION PICKER (full raw frame, very first thing)
    # =========================================================
    if ENABLE_ROTATION_PICKER:
        print("Step 1: Opening rotation picker on full raw frame...")
        rotation_angle = pick_rotation_angle(cap, ROI_SELECTION_FRAME, orig_w, orig_h)
    else:
        rotation_angle = FRAME_ROTATION_ANGLE
    print(f"Frame rotation angle: {rotation_angle:+.1f} degrees")

    # Pre-build the single rotation matrix used for every frame
    M_rot, rot_w, rot_h = build_rotation_matrix(orig_w, orig_h, rotation_angle)
    print(f"Original frame size : {orig_w}×{orig_h}")
    print(f"Rotated  frame size : {rot_w}×{rot_h}")

    # All subsequent geometry is in ROTATED frame space
    width, height = rot_w, rot_h

    display_scale  = get_display_scale(width, height, MAX_DISPLAY_WIDTH, MAX_DISPLAY_HEIGHT)
    display_width  = int(width  * display_scale)
    display_height = int(height * display_scale)

    # =========================================================
    # STEP 2 — ROI SELECTION (on the rotated frame)
    # =========================================================
    if ENABLE_MANUAL_ROI:
        print(f"\nStep 2: Opening rotated frame {ROI_SELECTION_FRAME} for ROI selection...")
        roi_x_min, roi_y_min, roi_w, roi_h_sel, roi_source_frame, roi_source_idx = \
            select_roi_from_rotated_frame(cap, ROI_SELECTION_FRAME, M_rot, rot_w, rot_h)
        roi_x_max = roi_x_min + roi_w
        roi_y_max = roi_y_min + roi_h_sel
        print(f"Selected ROI (rotated coords): x={roi_x_min}, y={roi_y_min}, "
              f"w={roi_w}, h={roi_h_sel}")
    else:
        roi_x_min = int(width  * ROI_X_MIN_RATIO)
        roi_x_max = int(width  * ROI_X_MAX_RATIO)
        roi_y_min = int(height * ROI_Y_MIN_RATIO)
        roi_y_max = int(height * ROI_Y_MAX_RATIO)
        # Get a reference rotated frame for the screenshot
        cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, min(ROI_SELECTION_FRAME, frame_count-1)))
        ok, _raw = cap.read()
        if not ok:
            raise RuntimeError("Could not read reference frame.")
        roi_source_frame = apply_rotation(_raw, M_rot, rot_w, rot_h)
        roi_source_idx   = ROI_SELECTION_FRAME

    line_x = int(width * LINE_POSITION_RATIO)

    save_roi_reference_image(roi_source_frame,
                             roi_x_min, roi_y_min, roi_x_max, roi_y_max,
                             line_x, roi_reference_path, roi_source_idx, rotation_angle)
    print(f"ROI reference image saved: {roi_reference_path}")

    # =========================================================
    # STEP 3 — FRAME RANGE
    # =========================================================
    if PROCESS_MODE == "full":
        start_frame = 0
        end_frame   = frame_count
        print(f"Processing full video ({duration:.1f}s, {frame_count} frames)")
    elif PROCESS_MODE == "frames":
        start_frame = max(0, int(START_FRAME))
        end_frame   = min(frame_count, int(END_FRAME))
        if start_frame >= end_frame:
            raise ValueError(f"Invalid range: START_FRAME={START_FRAME}, END_FRAME={END_FRAME}")
        print(f"Processing frames {start_frame}–{end_frame} of {frame_count}")
    else:
        raise ValueError("PROCESS_MODE must be 'full' or 'frames'")

    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    # =========================================================
    # STEP 4 — CSV
    # =========================================================
    csv_file   = open(output_csv, mode="w", newline="", encoding="utf-8")
    csv_writer = csv.DictWriter(csv_file, fieldnames=columns)
    csv_writer.writeheader()
    csv_file.flush()
    if FORCE_SAVE_EACH_LOG_TO_DISK:
        os.fsync(csv_file.fileno())
    print(f"Live CSV log: {os.path.abspath(output_csv)}")

    # Background subtractor + morphology kernel
    fgbg   = cv2.createBackgroundSubtractorMOG2(history=400,
                                                varThreshold=VAR_THRESHOLD,
                                                detectShadows=False)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

    # State
    next_id        = 1
    tracks         = {}
    log_count      = 0
    frame_idx      = start_frame
    lkw_cooldown   = 0
    lkw_logged     = False   # True while we are inside a single truck pass — prevents duplicate rows
    paused         = False
    mask_overlay_on = False
    playback_delay  = PLAYBACK_DELAY_MS
    recent_crossings = deque(maxlen=RECENT_CROSSINGS_SHOWN)
    last_vis        = None
    last_mask_full  = None
    current_frame   = start_frame

    window_name = "Vehicle Detection — Control Mode"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window_name, display_width, display_height)

    print("\nControls:")
    print("  SPACE — pause/resume  |  N — step one frame  |  +/- — speed")
    print("  M — mask overlay      |  S — snapshot         |  ESC — quit\n")

    # =======================
    # ---- PROCESS FRAME ----
    # =======================

    def process_frame(raw_frame, frame_idx):
        """
        1. Rotate the full raw frame.
        2. Crop ROI from the rotated frame.
        3. Run background subtraction + detection.
        4. All coordinates are in rotated-frame space.
        """
        global lkw_cooldown, lkw_logged, next_id, log_count

        t = frame_idx / max(fps, 1e-6)
        roi_h_current = roi_y_max - roi_y_min

        # --- 1. Rotate entire frame ---
        frame = apply_rotation(raw_frame, M_rot, rot_w, rot_h)

        # --- 2. Crop ROI (already in rotated space) ---
        roi_bgr  = frame[roi_y_min:roi_y_max, roi_x_min:roi_x_max]
        roi_gray = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2GRAY)

        # --- 3. Background subtraction ---
        lr   = 0.01 if lkw_cooldown > 0 else -1.0
        mask = fgbg.apply(roi_gray, learningRate=lr)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        mask = cv2.dilate(mask, kernel, iterations=DILATE_ITERS)
        _, mask = cv2.threshold(mask, 200, 255, cv2.THRESH_BINARY)

        mask_fill_ratio = np.count_nonzero(mask) / max(mask.size, 1)
        is_lkw_frame    = mask_fill_ratio > LKW_FILL_THRESHOLD

        if is_lkw_frame:
            # First saturation frame of this truck pass → log it immediately
            if not lkw_logged:
                lkw_logged = True
                # Estimate direction from any existing LKW track; fall back to "L2R"
                lkw_dx  = 0.0
                for tr in tracks.values():
                    if tr.vehicle_type == "LKW" and abs(tr.dx()) > 0.01:
                        lkw_dx = tr.dx();  break
                direction = crossing_label_from_dx(lkw_dx) if abs(lkw_dx) > 0.01 else "UNK"

                # Speed from the best available LKW track
                speed_kmh_lkw = 0.0
                for tr in tracks.values():
                    if tr.vehicle_type == "LKW" and abs(tr.dx()) > 0.01:
                        sp = abs(tr.dx() * (fps / PROCESS_EVERY_N_FRAMES) * METERS_PER_PIXEL) * 3.6
                        speed_kmh_lkw = min(sp, MAX_REALISTIC_SPEED_KMH);  break

                # row = {
                #     "TrackID":       -1,           # -1 = saturation-triggered, no single track
                #     "Frame":         frame_idx,
                #     "Timestamp_s":   round(t, 3),
                #     "Timestamp_hms": format_hms(t),
                #     "Direction":     direction,
                #     "VehicleType":   "LKW",
                #     "center_x":      -1,
                #     "y_center":      -1,
                #     "Speed_kmh":     round(speed_kmh_lkw, 2),
                # }
                row = {
                    "TrackID":       -1,
                    "Frame":         frame_idx,
                    "Timestamp_s":   round(t, 6),
                    "Timestamp_hms": format_hms(t),
                    "Timestamp_hms_us": f"{format_hms(t)}.{int((t % 1) * 1_000_000):06d}",
                    "Direction":     direction,
                    "VehicleType":   "LKW",
                    "center_x":      -1,
                    "y_center":      -1,
                    "Speed_kmh":     round(speed_kmh_lkw, 2),
                }
                
                write_log_row(csv_writer, csv_file, row)
                log_count += 1
                entry = f"LKW {direction} {speed_kmh_lkw:.0f}km/h {format_hms(t)} [SAT]"
                recent_crossings.appendleft(entry)
                print(f"Logged #{log_count}: LKW SATURATION {direction} "
                      f"{speed_kmh_lkw:.1f}km/h @ {format_hms(t)}")

            lkw_cooldown = LKW_COOLDOWN_FRAMES
            print(f"  [LKW] frame {frame_idx} — fill {mask_fill_ratio:.2f}")
        else:
            # Fill dropped back below threshold — reset so the next truck can be logged
            if lkw_cooldown > 0:
                lkw_cooldown -= 1
            if lkw_cooldown == 0:
                lkw_logged = False

        # Full-frame mask for overlay
        mask_full = np.zeros((rot_h, rot_w), dtype=np.uint8)
        mask_full[roi_y_min:roi_y_max, roi_x_min:roi_x_max] = mask

        # --- 4. Contour detection ---
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        detections  = []

        for c in contours:
            area    = cv2.contourArea(c)
            min_a   = LKW_MIN_CONTOUR_AREA if is_lkw_frame else MIN_CONTOUR_AREA
            if area < min_a:
                continue

            bx, by, bw, bh = cv2.boundingRect(c)
            # Shift from ROI-local → rotated-frame coords
            x, y, w, h = bx + roi_x_min, by + roi_y_min, bw, bh

            cx, cy       = box_center((x, y, w, h))
            aspect_ratio = w / float(max(h, 1))
            box_area     = w * h

            if is_lkw_frame:
                if w < width * LKW_MIN_BOX_WIDTH_RATIO:  continue
                if h < height* LKW_MIN_BOX_HEIGHT_RATIO: continue
                if w > width * LKW_MAX_BOX_WIDTH_RATIO:  continue
                if h > height* LKW_MAX_BOX_HEIGHT_RATIO: continue
                if not (LKW_MIN_ASPECT_RATIO <= aspect_ratio <= LKW_MAX_ASPECT_RATIO): continue
            else:
                if w < MIN_BOX_WIDTH or h < MIN_BOX_HEIGHT:          continue
                if box_area < MIN_BOX_AREA:                           continue
                if w > width * MAX_BOX_WIDTH_RATIO:                   continue
                if h > height* MAX_BOX_HEIGHT_RATIO:                  continue
                if ENABLE_ASPECT_RATIO_FILTER:
                    if not (MIN_ASPECT_RATIO <= aspect_ratio <= MAX_ASPECT_RATIO): continue
                if ENABLE_LOW_ZONE_FILTER:
                    if cy > roi_y_min + LOW_ZONE_Y_RATIO * roi_h_current: continue

            if not (roi_x_min <= cx <= roi_x_max and roi_y_min <= cy <= roi_y_max):
                continue

            detections.append((x, y, w, h))

        x_gap = LKW_MERGE_X_GAP if is_lkw_frame else MERGE_X_GAP
        y_gap = LKW_MERGE_Y_GAP if is_lkw_frame else MERGE_Y_GAP
        detections = merge_boxes(detections, x_gap, y_gap)

        # --- 5. Track association ---
        used = set()
        for tid, tr in list(tracks.items()):
            j = choose_detection_for_track(tr, detections, used)
            if j >= 0:
                tr.update(detections[j]);  used.add(j)
            else:
                tr.misses += 1
                if tr.misses > MAX_MISSES:
                    tracks.pop(tid, None)

        for j, det in enumerate(detections):
            if j not in used:
                vtype = "LKW" if is_lkw_frame else "PKW"
                tracks[next_id] = Track(next_id, det, vehicle_type=vtype)
                next_id += 1

        # --- 6. Crossing logic ---
        # LKW: logged once on the first saturation frame (above). Skip line logic for them.
        # PKW: standard leading-edge / centre crossing of line_x.
        for tid, tr in tracks.items():
            if tr.vehicle_type == "LKW":
                continue   # already handled by saturation trigger

            dx = tr.dx()
            if abs(dx) < 0.01:
                continue

            speed_mps = abs(dx * (fps / PROCESS_EVERY_N_FRAMES) * METERS_PER_PIXEL)
            speed_kmh = min(speed_mps * 3.6, MAX_REALISTIC_SPEED_KMH)

            if not tr.crossed and len(tr.box_hist) >= 2:
                prev_cx, _ = box_center(tr.box_hist[-2])
                curr_ref   = tr.center[0]   # PKW: use centre

                crossed_now = (dx > 0 and prev_cx < line_x <= curr_ref) or \
                              (dx < 0 and prev_cx > line_x >= curr_ref)

                if crossed_now:
                    tr.crossed = True
                    # row = {
                    #     "TrackID":      tid,
                    #     "Frame":        frame_idx,
                    #     "Timestamp_s":  round(t, 6),
                    #     "Timestamp_hms": format_hms(t),
                    #     "Timestamp_hms_us": f"{format_hms(t)}.{int((t % 1) * 1_000_000):06d}",
                    #     "Direction":    crossing_label_from_dx(dx),
                    #     "VehicleType":  tr.vehicle_type,
                    #     "center_x":     int(tr.center[0]),
                    #     "y_center":     int(tr.center[1]),
                    #     "Speed_kmh":    round(speed_kmh, 2),
                    # }
                    row = {
                    "TrackID":      tid,
                    "Frame":        frame_idx,
                    "Timestamp_s":  round(t, 6),
                    "Timestamp_hms": format_hms(t),
                    "Timestamp_hms_us": f"{format_hms(t)}.{int((t % 1) * 1_000_000):06d}",
                    "Direction":    crossing_label_from_dx(dx),
                    "VehicleType":  tr.vehicle_type,
                    "center_x":     int(tr.center[0]),
                    "y_center":     int(tr.center[1]),
                    "Speed_kmh":    round(speed_kmh, 2),
                    }
                    write_log_row(csv_writer, csv_file, row)
                    log_count += 1
                    entry = (f"{tr.vehicle_type} {crossing_label_from_dx(dx)} "
                             f"{speed_kmh:.0f}km/h {format_hms(t)}")
                    recent_crossings.appendleft(entry)
                    print(f"Logged #{log_count}: {tr.vehicle_type} ID={tid} "
                          f"{crossing_label_from_dx(dx)} {speed_kmh:.1f}km/h @ {format_hms(t)}")

        # --- 7. Visualisation (drawn on the rotated frame) ---
        vis       = frame.copy()
        roi_color = (0, 0, 255) if is_lkw_frame else (100, 255, 255)
        cv2.rectangle(vis, (roi_x_min, roi_y_min), (roi_x_max, roi_y_max), roi_color, 2)
        cv2.line(vis, (line_x, roi_y_min), (line_x, roi_y_max), (255, 0, 0), 2)

        for tid, tr in tracks.items():
            x, y, w, h = tr.box
            dx        = tr.dx()
            speed_mps = abs(dx * (fps / PROCESS_EVERY_N_FRAMES) * METERS_PER_PIXEL)
            speed_kmh = min(speed_mps * 3.6, MAX_REALISTIC_SPEED_KMH)
            color     = (0, 80, 255) if tr.vehicle_type == "LKW" else (0, 255, 0)

            cv2.rectangle(vis, (x, y), (x+w, y+h), color, 2)

            label = f"{tr.vehicle_type} ID{tid}"
            cv2.putText(vis, label, (x, max(18, y-22)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0,0,0), 3)
            cv2.putText(vis, label, (x, max(18, y-22)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2)

            spd_lbl = f"{speed_kmh:.1f} km/h"
            cv2.putText(vis, spd_lbl, (x, max(38, y-4)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0,0,0), 3)
            cv2.putText(vis, spd_lbl, (x, max(38, y-4)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0,255,255), 2)

            cx_tr    = int(x + w/2);  cy_tr = int(y + h/2)
            arr_len  = min(40, w//3)
            arr_dx   = arr_len if dx > 0 else -arr_len
            cv2.arrowedLine(vis, (cx_tr, cy_tr), (cx_tr+arr_dx, cy_tr),
                            color, 2, tipLength=0.4)

        return vis, mask_full, is_lkw_frame, mask_fill_ratio

    # =======================
    # ------ MAIN LOOP ------
    # =======================

    step_once = False

    while True:
        if frame_idx >= end_frame:
            print(f"End frame reached: {end_frame}")
            break

        advance    = (not paused) or step_once
        step_once  = False

        if advance:
            ok, raw_frame = cap.read()
            if not ok:
                print(f"cap.read() failed at frame {frame_idx}")
                break

            current_frame  = frame_idx
            frame_idx     += 1

            if current_frame % PROCESS_EVERY_N_FRAMES != 0:
                if last_vis is not None:
                    disp, _ = resize_for_display(last_vis, MAX_DISPLAY_WIDTH, MAX_DISPLAY_HEIGHT)
                    cv2.imshow(window_name, disp)
                if cv2.waitKey(1) == 27:
                    break
                continue

            t = current_frame / max(fps, 1e-6)
            vis, mask_full, is_lkw_frame, mask_fill_ratio = \
                process_frame(raw_frame, current_frame)
            last_vis       = vis.copy()
            last_mask_full = mask_full.copy()

        else:   # paused
            if last_vis is None:
                if cv2.waitKey(50) == 27:
                    break
                continue
            vis       = last_vis.copy()
            mask_full = last_mask_full if last_mask_full is not None \
                        else np.zeros((rot_h, rot_w), dtype=np.uint8)
            t = current_frame / max(fps, 1e-6)

        # Mask overlay
        if mask_overlay_on and mask_full is not None:
            mc = cv2.cvtColor(mask_full, cv2.COLOR_GRAY2BGR)
            mc[:, :, 0] = 0;  mc[:, :, 2] = 0
            vis = cv2.addWeighted(vis, 0.65, mc, 0.45, 0)

        draw_hud(vis, current_frame, end_frame, t, log_count,
                 paused, playback_delay, mask_overlay_on,
                 is_lkw_frame, mask_fill_ratio, lkw_cooldown,
                 recent_crossings, rot_w, rotation_angle)

        disp, _ = resize_for_display(vis, MAX_DISPLAY_WIDTH, MAX_DISPLAY_HEIGHT)
        cv2.imshow(window_name, disp)

        wait_ms = 50 if paused else max(1, playback_delay)
        key     = cv2.waitKey(wait_ms)

        if key == 27:
            print("ESC — stopping.")
            break
        elif cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            break
        elif key == ord(' '):
            paused = not paused
            print("Paused." if paused else "Resumed.")
        elif key in (ord('n'), ord('N')):
            if paused: step_once = True
        elif key in (ord('+'), ord('=')):
            playback_delay = max(1, playback_delay - 20)
            print(f"Speed: {1000/playback_delay:.1f} fps ({playback_delay} ms/frame)")
        elif key == ord('-'):
            playback_delay = min(2000, playback_delay + 20)
            print(f"Speed: {1000/playback_delay:.1f} fps ({playback_delay} ms/frame)")
        elif key in (ord('m'), ord('M')):
            mask_overlay_on = not mask_overlay_on
            print(f"Mask overlay: {'ON' if mask_overlay_on else 'OFF'}")
        elif key in (ord('s'), ord('S')):
            snap = os.path.join(video_dir, f"snapshot_frame_{current_frame}.jpg")
            cv2.imwrite(snap, vis)
            print(f"Snapshot saved: {snap}")

    print(f"\nLog saved: {os.path.abspath(output_csv)} ({log_count} crossings)")

finally:
    cap.release()
    cv2.destroyAllWindows()
    if csv_file is not None:
        csv_file.flush()
        if FORCE_SAVE_EACH_LOG_TO_DISK:
            os.fsync(csv_file.fileno())
        csv_file.close()

## Obtain Frames

In [ ]:
import os
import cv2
import pandas as pd

# --- Paths ---
csv_path = r"xxx.csv"
video_path = r"xxx.mp4"
output_dir = r"xxx"

# --- Create output directory if it doesn't exist ---
os.makedirs(output_dir, exist_ok=True)

# --- Load CSV ---
df = pd.read_csv(csv_path)
df = pd.read_csv(csv_path, sep=";")
# Optional: strip spaces from column names (just in case)
# df.columns = df.columns.str.strip()

# Ensure "Frame" column exists
if "Frame" not in df.columns:
    print("Columns found:", df.columns.tolist())
    raise ValueError("CSV does not contain a 'Frame' column")


# Get unique frame numbers (avoid duplicates)
frames_to_extract = sorted(df["Frame"].dropna().astype(int).unique())
# frames_to_extract = sorted((df["Frame"] - 1).dropna().astype(int).unique())

# --- Open video ---
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise IOError("Cannot open video file")

# --- Extract frames ---
sequence_counter = 1

for frame_number in frames_to_extract:
    # Set video to the desired frame
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)

    ret, frame = cap.read()

    if not ret:
        print(f"Warning: Could not read frame {frame_number}")
        continue

    # Build filename
    filename = f"united_2_sc_{sequence_counter}_frame_{frame_number}.jpg"
    output_path = os.path.join(output_dir, filename)

    # Save image
    cv2.imwrite(output_path, frame)

    print(f"Saved: {output_path}")

    sequence_counter += 1

# --- Release video ---
cap.release()

print("Done.")